# Feature engineering of Smart Card data from Santiago, Chile for baseline and Machine Learning model implementations

Last reviewed: Monday 18-05-2026

Times this file has been edited: 3

Hardware specs:


1. *Portable* branch:
    * Machine: MacBook Air (13-inch, 2017)
    * OS: MacOS Monterey v12.7.6
    * CPU: 1.8 GHz Intel Core i5 de dos núcleos
    * RAM: 8 GB 1600 MHz DDR3
    * Graphics: Intel HD Graphics 6000 1536 MB

2. *House* branch:
    * OS: Windows 10 Home 64-bit (10.0, Build 19045)
    * Processor: Intel(R) Core(TM) i7-8700K CPU @ 3.70GHz (12 CPUs), ~3.7GHz
    * Memory: 12288MB RAM
    * GPU: NVIDIA GeForce RTX 3060 12115 MB

## Script 01: Construcción de dataset para primera versión de modelo con datasets de viajes 2024 y 2025

Los pasos son:

1. Enriquecer diccionario_paraderos con celdas H3
2. Filtrar trips a punta mañana (06:30–09:00), 2024 + 2025
3. Asignar H3 de origen y destino a cada viaje
4. Definir ruta_id como concatenación de srv_1|srv_2|srv_3
5. Calcular perfil de cada ruta por OD: atributos promedio
6. Filtrar: $\geq 5$ observaciones por ruta, $\geq 2$ rutas por OD
7. Generar formato largo con elegida = 1 para la ruta tomada
8. Añadir variables de contexto: clima, tipo de día, período

In [10]:
"""
Script 01: Feature engineering - construcción del dataset de modelado

Output: data/parquets/04_features/dataset_modelado.parquet
Formato: largo (una fila por viaje \times alternativa de ruta)

Decisiones:
  - Período:        punta mañana 06:30-09:00
  - Datos:          Trips 2024 + 2025 combinados
  - OD:             hexágonos H3 resolución 8 (~460m radio)
  - Ruta:           concatenación srv_1|srv_2|srv_3
  - Threshold:      ≥5 observaciones por ruta por OD
  - OD mínimo:      ≥2 rutas alternativas
  - Atributos ruta: promedio observado de trips que tomaron esa ruta en ese OD
"""

import gc
import duckdb
import pandas as pd
import numpy as np
import h3
from pathlib import Path
from pyproj import Transformer

# ── Rutas ────────────────────────────────────────────────────────────────────
BASE_DIR     = Path(r"D:\GitHub\tesis_magister_route_choice_modelling\data")
PARQUETS_DIR = BASE_DIR / "parquets"
CLEAN_DIR    = PARQUETS_DIR / "03_clean"
ENRICHED_DIR = PARQUETS_DIR / "02_enriched"
FEATURES_DIR = PARQUETS_DIR / "04_features"

TRIPS_2024   = CLEAN_DIR / "Trips" / "2024" / "trips_2024_clean.parquet"
TRIPS_2025   = CLEAN_DIR / "Trips" / "2025" / "trips_2025_clean.parquet"
CLIMA_DIR    = ENRICHED_DIR / "clima"
GTFS_DIR     = ENRICHED_DIR / "gtfs"
DICT_PARAD   = GTFS_DIR / "diccionario_paraderos.parquet"

# Parámetros
H3_RESOLUTION  = 8
MIN_OBS_RUTA   = 5     # mínimo de observaciones para incluir una ruta
MIN_RUTAS_OD   = 2     # mínimo de rutas alternativas por OD
HORA_INI       = "06:30:00"
HORA_FIN       = "09:00:00"

TRANSFORMER = Transformer.from_crs("EPSG:32719", "EPSG:4326", always_xy=True)


def asegurar_carpeta(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def crear_conexion() -> duckdb.DuckDBPyConnection:
    con = duckdb.connect()
    con.execute("SET memory_limit = '3GB'")
    con.execute("SET threads = 6")
    con.execute("SET temp_directory = 'D:/temp_duckdb'")
    return con


# ── Paso 1: Enriquecer diccionario de paraderos con H3 ───────────────────────

def enriquecer_diccionario_h3() -> pd.DataFrame:
    """
    Añade columna h3_cell al diccionario de paraderos.
    Guarda versión enriquecida para reutilización.
    """
    salida = GTFS_DIR / "diccionario_paraderos_h3.parquet"

    if salida.exists():
        print("  [CACHE] diccionario_paraderos_h3.parquet ya existe")
        return pd.read_parquet(salida)

    print("  Calculando H3 para diccionario de paraderos...")
    df = pd.read_parquet(DICT_PARAD)

    def safe_h3(row):
        try:
            if pd.notna(row["lat"]) and pd.notna(row["lon"]):
                return h3.latlng_to_cell(row["lat"], row["lon"], H3_RESOLUTION)
            return None
        except Exception:
            return None

    df["h3_cell"] = df.apply(safe_h3, axis=1)

    n_con_h3 = df["h3_cell"].notna().sum()
    print(f"  Paraderos con H3 asignado: {n_con_h3:,} / {len(df):,}")

    df.to_parquet(salida, compression="zstd", index=False)
    return df


# ── Paso 2: Cargar y filtrar trips a punta mañana ────────────────────────────

def cargar_trips_punta_manana(
    df_dict: pd.DataFrame,
    con: duckdb.DuckDBPyConnection
) -> pd.DataFrame:
    """
    Carga Trips 2024 + 2025, filtra a punta mañana y asigna H3 de origen/destino.
    Solo viajes con destino conocido (flag_destino_conocido = 1).
    Solo días laborales (tipodia = 0; en trips: 0=LABORAL, 1=SÁBADO, 2=DOMINGO).
    """
    trips_2024 = str(TRIPS_2024).replace("\\", "/")
    trips_2025 = str(TRIPS_2025).replace("\\", "/")

    print("  Cargando trips punta mañana (06:30-09:00)...")

    df_trips = con.sql(f"""
        SELECT
            -- Identificación del viaje
            id_tarjeta,
            tiempo_inicio_viaje,
            CAST(tiempo_inicio_viaje AS DATE)::VARCHAR         AS fecha,
            CAST(tiempo_inicio_viaje AS TIME)::VARCHAR         AS hora_inicio,

            -- Atributos del viaje
            n_etapas,
            tipodia,
            modos,
            periodo_inicio_viaje,

            -- Servicios por etapa (definen la ruta)
            srv_1, srv_2, srv_3,

            -- Paraderos de origen y destino
            paradero_inicio_viaje,
            paradero_fin_viaje,

            -- Atributos de tiempo (segundos)
            tviaje_calculado,
            te0,  tv1,  tc1,
            te1,  tv2,  tc2,
            te2,  tv3,

            -- Atributos de distancia (metros)
            dtfinal,
            dveh_rutafinal,
            distancia_eucl,
            distancia_ruta,

            -- Flags y factores
            flag_destino_conocido,
            factor_expansion,

            -- Clima: fecha y mediahora para join
            CAST(mediahora_inicio_viaje AS INTEGER)            AS mediahora_num

        FROM (
            SELECT *, '2024' AS anio
            FROM read_parquet('{trips_2024}')
            UNION ALL
            SELECT *, '2025' AS anio
            FROM read_parquet('{trips_2025}')
        )
        WHERE
            -- Solo punta mañana
            CAST(tiempo_inicio_viaje AS TIME)
                BETWEEN TIME '{HORA_INI}' AND TIME '{HORA_FIN}'
            -- Solo días laborales (0=LABORAL, 1=SÁBADO, 2=DOMINGO en trips)
            AND CAST(tipodia AS INTEGER) = 0
            -- Solo viajes con destino conocido
            AND flag_destino_conocido = 1
            -- Descartar viajes sin srv_1
            AND srv_1 IS NOT NULL
            AND TRIM(srv_1) != ''
    """).df()

    print(f"  Trips cargados: {len(df_trips):,}")

    # Definir ruta_id como concatenación de servicios no nulos
    def construir_ruta_id(row):
        partes = []
        for col in ["srv_1", "srv_2", "srv_3"]:
            v = row.get(col)
            if pd.notna(v) and str(v).strip() not in ("", "-", "None"):
                partes.append(str(v).strip())
        return "|".join(partes) if partes else None

    df_trips["ruta_id"] = df_trips.apply(construir_ruta_id, axis=1)

    # Unir con diccionario para obtener H3 de origen
    dict_h3 = df_dict[["parada_bip", "h3_cell"]].rename(
        columns={"parada_bip": "paradero_inicio_viaje",
                 "h3_cell":    "h3_origen"}
    )
    df_trips = df_trips.merge(dict_h3, on="paradero_inicio_viaje", how="left")

    # H3 de destino
    dict_h3_dest = df_dict[["parada_bip", "h3_cell"]].rename(
        columns={"parada_bip": "paradero_fin_viaje",
                 "h3_cell":    "h3_destino"}
    )
    df_trips = df_trips.merge(dict_h3_dest, on="paradero_fin_viaje", how="left")

    # Filtrar viajes con OD H3 completo
    n_antes = len(df_trips)
    df_trips = df_trips.dropna(subset=["h3_origen", "h3_destino", "ruta_id"])
    n_despues = len(df_trips)

    print(f"  Tras asignar H3 y ruta_id: {n_despues:,} ({n_antes - n_despues:,} descartados)")

    return df_trips


# ── Paso 3: Construir perfiles de ruta por OD ────────────────────────────────

def construir_perfiles_ruta(df_trips: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada (h3_origen, h3_destino, ruta_id), calcula:
    - Atributos promedio de todos los trips que tomaron esa ruta en ese OD
    - Número de observaciones
    """
    print("  Construyendo perfiles de ruta por OD H3...")

    atributos = [
        "tviaje_calculado",
        "te0", "tv1", "tc1",
        "te1", "tv2", "tc2",
        "te2", "tv3",
        "dtfinal",
        "dveh_rutafinal",
        "distancia_eucl",
        "distancia_ruta",
    ]

    agg = {col: "mean" for col in atributos if col in df_trips.columns}
    agg["id_tarjeta"] = "count"

    df_perfiles = (
        df_trips
        .groupby(["h3_origen", "h3_destino", "ruta_id"])
        .agg(agg)
        .reset_index()
    )
    df_perfiles = df_perfiles.rename(columns={"id_tarjeta": "n_obs_ruta"})

    cols_espera = [c for c in ["te0", "te1", "te2"] if c in df_perfiles.columns]
    if cols_espera:
        df_perfiles["tiempo_espera_total"] = df_perfiles[cols_espera].sum(axis=1, skipna=True)

    cols_invehiculo = [c for c in ["tv1", "tv2", "tv3"] if c in df_perfiles.columns]
    if cols_invehiculo:
        df_perfiles["tiempo_invehiculo_total"] = df_perfiles[cols_invehiculo].sum(axis=1, skipna=True)

    cols_caminata = [c for c in ["tc1", "tc2"] if c in df_perfiles.columns]
    if cols_caminata:
        df_perfiles["tiempo_caminata_total"] = df_perfiles[cols_caminata].sum(axis=1, skipna=True)

    print(f"  Perfiles generados: {len(df_perfiles):,} rutas únicas por OD")
    return df_perfiles


# ── Paso 4: Filtrar y construir choice sets ───────────────────────────────────

def construir_choice_sets(df_perfiles: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Filtra por thresholds y construye los choice sets válidos.
    Retorna:
      - df_perfiles_filtrado: perfiles que cumplen los criterios
      - stats: estadísticas del proceso de filtrado
    """
    print(f"  Filtrando choice sets (≥{MIN_OBS_RUTA} obs/ruta, ≥{MIN_RUTAS_OD} rutas/OD)...")

    n_rutas_antes = len(df_perfiles)
    n_od_antes    = df_perfiles.groupby(["h3_origen", "h3_destino"]).ngroups

    df_f = df_perfiles[df_perfiles["n_obs_ruta"] >= MIN_OBS_RUTA].copy()

    n_rutas_por_od = df_f.groupby(["h3_origen", "h3_destino"])["ruta_id"].transform("count")
    df_f = df_f[n_rutas_por_od >= MIN_RUTAS_OD].copy()

    n_rutas_despues = len(df_f)
    n_od_despues    = df_f.groupby(["h3_origen", "h3_destino"]).ngroups

    stats = {
        "rutas_antes":   n_rutas_antes,
        "rutas_despues": n_rutas_despues,
        "od_antes":      n_od_antes,
        "od_despues":    n_od_despues,
    }

    print(f"  Rutas: {n_rutas_antes:,} → {n_rutas_despues:,}")
    print(f"  OD pairs: {n_od_antes:,} → {n_od_despues:,}")

    dist = (
        df_f.groupby(["h3_origen", "h3_destino"])["ruta_id"]
        .count()
        .value_counts()
        .sort_index()
    )
    print("\n  Distribución de alternativas por OD pair:")
    for n_alt, n_od in dist.items():
        print(f"    {n_alt} alternativas: {n_od:,} OD pairs")

    return df_f, stats


# ── Paso 5: Generar dataset en formato largo ──────────────────────────────────

def generar_formato_largo(
    df_trips: pd.DataFrame,
    df_perfiles: pd.DataFrame
) -> pd.DataFrame:
    """
    Para cada viaje en la punta mañana:
      - Busca su OD pair en el choice set
      - Si la ruta que tomó está en el choice set, genera una fila por alternativa
      - elegida = 1 para la ruta tomada, 0 para las demás

    Solo incluye viajes cuya ruta efectiva está en el choice set.
    """
    print("  Generando dataset en formato largo...")

    # Filtrar trips a OD pairs válidos (merge vectorizado, robusto a DFs vacíos)
    od_valid_df = df_perfiles[["h3_origen", "h3_destino"]].drop_duplicates()
    df_trips_validos = df_trips.merge(
        od_valid_df, on=["h3_origen", "h3_destino"], how="inner"
    ).copy()

    # Verificar que la ruta tomada está en el choice set
    cs_df = df_perfiles[["h3_origen", "h3_destino", "ruta_id"]].drop_duplicates()
    df_trips_validos = df_trips_validos.merge(
        cs_df, on=["h3_origen", "h3_destino", "ruta_id"], how="inner"
    ).copy()

    n_viajes = len(df_trips_validos)
    print(f"  Viajes con ruta en choice set: {n_viajes:,}")

    df_trips_validos["id_obs"] = (
        df_trips_validos["id_tarjeta"].astype(str) + "_" +
        df_trips_validos["fecha"].astype(str) + "_" +
        df_trips_validos["hora_inicio"].astype(str)
    )

    cols_viaje = [
        "id_obs", "id_tarjeta", "fecha", "hora_inicio",
        "h3_origen", "h3_destino",
        "ruta_id",
        "tipodia", "periodo_inicio_viaje", "mediahora_num",
        "factor_expansion",
    ]

    df_viajes_cs = df_trips_validos[cols_viaje].merge(
        df_perfiles[
            ["h3_origen", "h3_destino", "ruta_id", "n_obs_ruta"] +
            [c for c in df_perfiles.columns
             if c not in ["h3_origen", "h3_destino", "ruta_id", "n_obs_ruta"]]
        ].rename(columns={"ruta_id": "ruta_alternativa"}),
        on=["h3_origen", "h3_destino"],
        how="inner"
    )

    df_viajes_cs["elegida"] = (
        df_viajes_cs["ruta_id"] == df_viajes_cs["ruta_alternativa"]
    ).astype(int)

    elecciones_por_obs = df_viajes_cs.groupby("id_obs")["elegida"].sum()
    sin_eleccion  = (elecciones_por_obs == 0).sum()
    con_mas_de_1  = (elecciones_por_obs > 1).sum()

    if sin_eleccion > 0 or con_mas_de_1 > 0:
        print(f"  ⚠ Obs sin elección: {sin_eleccion} | Obs con >1 elegida: {con_mas_de_1}")
    else:
        print(f"  ✓ Estructura de elección válida en todas las observaciones")

    print(f"  Filas totales (viajes \times alternativas): {len(df_viajes_cs):,}")
    print(f"  Observaciones únicas (viajes):          {df_viajes_cs['id_obs'].nunique():,}")

    return df_viajes_cs


# ── Paso 6: Añadir variables de contexto (streaming a disco) ─────────────────

def añadir_contexto_clima(
    df: pd.DataFrame,
    con: duckdb.DuckDBPyConnection,
    salida: Path,
) -> None:
    """
    Une variables climáticas al dataset largo y escribe directamente a parquet.

    Estrategia anti-OOM: en lugar de registrar el DataFrame de 18M filas en
    DuckDB y materializar el join en RAM, se usa DuckDB COPY (streaming):
      1. df se vuelca a un parquet temporal en disco
      2. DuckDB lee el temporal + clima y escribe 'salida' sin cargar el
         resultado completo en RAM (usa temp_directory para spill si necesita)
    El caller debe hacer `del df; gc.collect()` después de llamar esta función.
    """
    temp_path  = FEATURES_DIR / "_temp_largo.parquet"
    clima_path = str(CLIMA_DIR / "clima_todos_periodos.parquet").replace("\\", "/")
    salida_str = str(salida).replace("\\", "/")
    temp_str   = str(temp_path).replace("\\", "/")

    print("  Escribiendo dataset intermedio a disco (evita OOM en join)...")
    df.to_parquet(temp_path, compression="zstd", index=False)

    print("  Ejecutando join climático via DuckDB streaming → parquet...")
    con.sql(f"""
        COPY (
            SELECT
                m.*,
                c.temperature_2m,
                c.llueve,
                c.lluvia_intensa,
                c.condicion       AS condicion_climatica,
                c.wind_speed_10m,
                c.relative_humidity_2m
            FROM read_parquet('{temp_str}') m
            LEFT JOIN (
                SELECT fecha, mediahora, temperature_2m, llueve, lluvia_intensa,
                       condicion, wind_speed_10m, relative_humidity_2m
                FROM read_parquet('{clima_path}')
            ) c
                ON  m.fecha         = c.fecha
                AND (CAST(m.mediahora_num AS INTEGER) / 2) * 2 = c.mediahora
        ) TO '{salida_str}' (FORMAT PARQUET, COMPRESSION 'zstd')
    """)

    temp_path.unlink(missing_ok=True)

    cobertura = con.sql(f"""
        SELECT 100.0 * SUM(CASE WHEN temperature_2m IS NOT NULL THEN 1 ELSE 0 END)
                     / COUNT(*) AS cob
        FROM read_parquet('{salida_str}')
    """).fetchone()[0]
    print(f"  Cobertura climática: {cobertura:.1f}%")

In [11]:
# ── Diagnóstico de exclusiones del choice set ────────────────────────────────

def diagnosticar_exclusiones(
    df_trips: pd.DataFrame,
    df_perfiles_filtrado: pd.DataFrame,
) -> None:
    """
    Clasifica los viajes descartados durante la construcción del choice set.

    Dos motivos de exclusión:
      A) El OD pair no alcanzó ≥MIN_RUTAS_OD rutas con ≥MIN_OBS_RUTA obs
         → choice set inválido; el OD entero se descarta.
      B) El OD pair tiene choice set válido, pero la ruta tomada en ese viaje
         específico no superó el umbral de observaciones.

    Debe llamarse después de construir_choice_sets y antes de liberar df_trips.
    """
    print("  Clasificando motivos de exclusión...")

    n_total = len(df_trips)

    od_validos = df_perfiles_filtrado[["h3_origen", "h3_destino"]].drop_duplicates()
    df_od_ok   = df_trips.merge(od_validos, on=["h3_origen", "h3_destino"], how="inner")
    n_od_ok    = len(df_od_ok)

    rutas_en_cs = df_perfiles_filtrado[["h3_origen", "h3_destino", "ruta_id"]].drop_duplicates()
    n_incluidos = len(
        df_od_ok.merge(rutas_en_cs, on=["h3_origen", "h3_destino", "ruta_id"], how="inner")
    )

    n_excl_od   = n_total - n_od_ok
    n_excl_ruta = n_od_ok - n_incluidos

    def pct(n):
        return 100.0 * n / n_total

    print(f"\n  Universo punta mañana (con H3 + ruta_id):   {n_total:>10,}   [100.0%]")
    print(f"  ├─ ✓ Incluidos en dataset de modelado:       {n_incluidos:>10,}   [{pct(n_incluidos):5.1f}%]")
    print(f"  ├─ ✗ Motivo A — OD sin choice set válido:    {n_excl_od:>10,}   [{pct(n_excl_od):5.1f}%]")
    print(f"  │       (OD pair con <{MIN_RUTAS_OD} rutas que alcancen ≥{MIN_OBS_RUTA} observaciones)")
    print(f"  └─ ✗ Motivo B — Ruta propia no en CS:        {n_excl_ruta:>10,}   [{pct(n_excl_ruta):5.1f}%]")
    print(f"          (OD válido, pero la ruta del viaje tiene <{MIN_OBS_RUTA} obs en ese OD)")

In [12]:
# ── Ejecución principal ───────────────────────────────────────────────────────

if __name__ == "__main__":

    asegurar_carpeta(FEATURES_DIR)

    print("=" * 60)
    print("FEATURE ENGINEERING — DATASET DE MODELADO")
    print("=" * 60)

    # Paso 1: Diccionario con H3
    print("\n[1/6] Enriqueciendo diccionario de paraderos con H3...")
    df_dict = enriquecer_diccionario_h3()

    # Paso 2: Cargar trips punta mañana
    print("\n[2/6] Cargando trips punta mañana...")
    con = crear_conexion()
    df_trips = cargar_trips_punta_manana(df_dict, con)
    del df_dict
    gc.collect()

    # Paso 3: Perfiles de ruta
    print("\n[3/6] Construyendo perfiles de ruta...")
    df_perfiles = construir_perfiles_ruta(df_trips)

    # Paso 4: Choice sets
    print("\n[4/6] Construyendo choice sets...")
    df_perfiles_filtrado, stats = construir_choice_sets(df_perfiles)
    del df_perfiles
    gc.collect()

    # Diagnóstico de exclusiones — df_trips y df_perfiles_filtrado aún en memoria
    print("\n[Diagnóstico] Razones de exclusión del choice set...")
    diagnosticar_exclusiones(df_trips, df_perfiles_filtrado)

    # Paso 5: Formato largo
    print("\n[5/6] Generando formato largo...")
    df_largo = generar_formato_largo(df_trips, df_perfiles_filtrado)
    del df_trips, df_perfiles_filtrado  # liberar RAM antes del join climático
    gc.collect()

    # Paso 6: Contexto climático — streaming a disco para evitar OOM
    # añadir_contexto_clima escribe directamente al parquet final; no retorna DataFrame.
    print("\n[6/6] Añadiendo variables de contexto...")
    salida = FEATURES_DIR / "dataset_modelado.parquet"
    añadir_contexto_clima(df_largo, con, salida)
    del df_largo
    gc.collect()
    con.close()

    # Resumen final — leer stats desde disco sin cargar 18M filas en pandas
    salida_str = str(salida).replace("\\", "/")
    con2 = crear_conexion()

    n_filas, n_obs, n_od = con2.sql(f"""
        SELECT
            COUNT(*)                                              AS n_filas,
            COUNT(DISTINCT id_obs)                               AS n_obs,
            COUNT(DISTINCT (h3_origen || '-' || h3_destino))     AS n_od
        FROM read_parquet('{salida_str}')
    """).fetchone()

    # Leer 0 filas para obtener nombres de columnas sin cargar datos en RAM
    columnas = sorted(
        con2.sql(f"SELECT * FROM read_parquet('{salida_str}') LIMIT 0")
        .df().columns.tolist()
    )
    con2.close()

    print("\n" + "=" * 60)
    print("✓ DATASET DE MODELADO GENERADO")
    print("=" * 60)
    print(f"  Archivo:          {salida.name}")
    print(f"  Tamaño:           {salida.stat().st_size / 1e6:.1f} MB")
    print(f"  Filas totales:    {n_filas:,}")
    print(f"  Observaciones:    {n_obs:,}")
    print(f"  OD pairs:         {stats['od_despues']:,}")
    print(f"  Columnas:         {len(columnas)}")
    print(f"\n  Columnas del dataset:")
    for col in columnas:
        print(f"    - {col}")

FEATURE ENGINEERING — DATASET DE MODELADO

[1/6] Enriqueciendo diccionario de paraderos con H3...
  [CACHE] diccionario_paraderos_h3.parquet ya existe

[2/6] Cargando trips punta mañana...
  Cargando trips punta mañana (06:30-09:00)...
  Trips cargados: 6,213,641
  Tras asignar H3 y ruta_id: 6,047,156 (166,485 descartados)

[3/6] Construyendo perfiles de ruta...
  Construyendo perfiles de ruta por OD H3...
  Perfiles generados: 993,092 rutas únicas por OD

[4/6] Construyendo choice sets...
  Filtrando choice sets (≥5 obs/ruta, ≥2 rutas/OD)...
  Rutas: 993,092 → 111,152
  OD pairs: 276,147 → 30,979

  Distribución de alternativas por OD pair:
    2 alternativas: 13,963 OD pairs
    3 alternativas: 6,485 OD pairs
    4 alternativas: 3,694 OD pairs
    5 alternativas: 2,258 OD pairs
    6 alternativas: 1,457 OD pairs
    7 alternativas: 987 OD pairs
    8 alternativas: 699 OD pairs
    9 alternativas: 457 OD pairs
    10 alternativas: 273 OD pairs
    11 alternativas: 216 OD pairs
    12 

Se agregarán un par de features adicionales para tener conocimiento del número de transbordos y si lleva metro

In [18]:
import duckdb
from pathlib import Path

FEATURES_DIR = Path(
    r"D:\GitHub\tesis_magister_route_choice_modelling\data\parquets\04_features"
)

entrada = str(FEATURES_DIR / "dataset_modelado.parquet").replace("\\", "/")
salida  = str(FEATURES_DIR / "dataset_modelado_v2.parquet").replace("\\", "/")

con = duckdb.connect()
con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 6")
con.execute("SET temp_directory = 'D:/temp_duckdb'")

print("Añadiendo features finales al dataset...")

con.execute(f"""
    COPY (
        SELECT
            *,

            -- Número de transbordos de la alternativa evaluada
            -- (cuenta los separadores | en ruta_alternativa)
            LEN(ruta_alternativa) - LEN(REPLACE(ruta_alternativa, '|', ''))
                AS n_transbordos,

            -- Indica si la alternativa incluye Metro
            -- (los servicios de Metro contienen letras L seguidas de número)
            CASE
                WHEN REGEXP_MATCHES(ruta_alternativa, '.*L[0-9].*')
                THEN 1 ELSE 0
            END AS tiene_metro,

            -- Tiempo de espera en minutos (para interpretabilidad)
            ROUND(tiempo_espera_total    / 60.0, 2) AS tiempo_espera_min,
            ROUND(tiempo_invehiculo_total / 60.0, 2) AS tiempo_invehiculo_min,
            ROUND(tiempo_caminata_total  / 60.0, 2) AS tiempo_caminata_min,
            ROUND(tviaje_calculado       / 60.0, 2) AS tviaje_min,

            -- Distancia de caminata en transbordo en km
            ROUND(dtfinal / 1000.0, 4) AS dist_caminata_km

        FROM read_parquet('{entrada}')
    )
    TO '{salida}'
    (FORMAT PARQUET, COMPRESSION 'ZSTD', ROW_GROUP_SIZE 100000)
""")

# Verificar distribución de n_transbordos por alternativa
print("\nDistribución n_transbordos por alternativa:")
con.sql(f"""
    SELECT
        n_transbordos,
        COUNT(*)     AS n_filas,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM read_parquet('{salida}')
    GROUP BY n_transbordos
    ORDER BY n_transbordos
""").show()

print("\nDistribución tiene_metro por alternativa:")
con.sql(f"""
    SELECT
        tiene_metro,
        COUNT(*)     AS n_filas,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM read_parquet('{salida}')
    GROUP BY tiene_metro
    ORDER BY tiene_metro
""").show()

n_cols = con.sql(f"SELECT * FROM read_parquet('{salida}') LIMIT 0").df().shape[1]
print(f"\nColumnas finales: {n_cols}")

con.close()
print("\n✓ dataset_modelado_v2.parquet generado")

Añadiendo features finales al dataset...

Distribución n_transbordos por alternativa:
┌───────────────┬──────────┬────────┐
│ n_transbordos │ n_filas  │  pct   │
│     int64     │  int64   │ double │
├───────────────┼──────────┼────────┤
│             0 │ 11327119 │   63.1 │
│             1 │  6581477 │   36.7 │
│             2 │    46751 │    0.3 │
└───────────────┴──────────┴────────┘


Distribución tiene_metro por alternativa:
┌─────────────┬─────────┬────────┐
│ tiene_metro │ n_filas │  pct   │
│    int32    │  int64  │ double │
├─────────────┼─────────┼────────┤
│           0 │ 9625273 │   53.6 │
│           1 │ 8330074 │   46.4 │
└─────────────┴─────────┴────────┘


Columnas finales: 43

✓ dataset_modelado_v2.parquet generado
